# 2. — ASD prevalence ETL (OECD cohort)

## Objective

This notebook performs the ETL process required to transform the raw
GBD ASD prevalence dataset into a clean analytical dataset.

Input datasets:

- `data/1_raw/ihme_gbd_asd_prevalence_oecd_1990_2023.csv`
- `data/3_processed/oecd_members.csv`

Main steps:

1. Load raw GBD dataset
2. Standardize column names
3. Normalize categorical variables
4. Validate cohort membership
5. Export cleaned dataset

## 2.1 Environment and project setup

This section prepares the execution environment for the notebook.

It includes:
- core library imports
- environment checks
- project path configuration

In [3]:
# --- Imports ---

import pandas as pd
import sys
from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, ensure_data_dirs

In [4]:
# --- Environment check ---

print("Python version:", sys.version.split()[0])

try:
    import src
    print("Project package import: OK")
except ImportError:
    print("ERROR: project package 'src' not found")

try:
    import pandas as pd
    print("pandas version:", pd.__version__)
except ImportError:
    print("ERROR: pandas not installed")

Python version: 3.13.5
Project package import: OK
pandas version: 2.2.3


In [5]:
# --- Project paths configuration ---

ensure_data_dirs()

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw
PROCESSED_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed


## 2.2 Data ingestion

This section loads the raw ASD prevalence dataset obtained from the Global Burden of Disease (IHME) database.

The objective is to ingest the dataset from the project's raw data directory and ensure that the expected input file is available before loading it into memory.

Key steps:
- locate the raw dataset in the project data directory
- verify that the dataset file exists
- load the dataset into a pandas DataFrame

In [6]:
# --- Locate raw ASD prevalence dataset ---

# Define the expected raw dataset filename
raw_filename = "ihme_gbd_asd_prevalence_oecd_1990_2023.csv"

# Build full path to the raw dataset
raw_path = RAW_DIR / raw_filename

# --- Verify if dataset file exists ---

# Ensure the file exists before proceeding
assert raw_path.exists(), f"Raw dataset not found: {raw_path}"

# --- Load raw ASD prevalence dataset ---

# Read raw dataset from the project raw data directory
df_raw = pd.read_csv(raw_path)

print("Raw dataset located:", raw_path)

Raw dataset located: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw\ihme_gbd_asd_prevalence_oecd_1990_2023.csv


## 2.3 Data transformation

In this section I begin the ETL process required to prepare the analytical dataset.

The objective is to inspect the raw GBD dataset structure and transform it into a clean, consistent and suitable dataframe for analysis.

Key steps:
- create a working copy of the raw dataset
- perform a basic structural summary of the dataset
- inspect categorical variables before normalization
- normalize categorical variables
- rename analytical measure columns for clarity
- prepare the dataset structure for downstream analysis

In [7]:
# --- Create a working dataframe for the ETL process ---

df = df_raw.copy()

In [8]:
# --- BASIC EDA: Create a function which return a structural summary to explore the dataset before transformation ---

def basic_eda_summary(df):

    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "null_count": df.isnull().sum(),
        "duplicate_count": [df[col].duplicated().sum() for col in df.columns],
        "n_unique": df.nunique(),
        "unique_values": [df[col].dropna().unique() for col in df.columns]
    })

    print("DATASET SHAPE")
    print("-------------")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}\n")

    return summary

In [9]:
eda_summary = basic_eda_summary(df)
eda_summary

DATASET SHAPE
-------------
Rows: 38760
Columns: 18



,dtype,null_count,duplicate_count,n_unique,unique_values
population_group_id,int64,0,38759,1,[1]
population_group_name,object,0,38759,1,[Toda la población]
measure_id,int64,0,38759,1,[5]
measure_name,object,0,38759,1,[Prevalencia]
location_id,int64,0,38722,38,"[55, 155, 98, 125, 67, 89, 75, 90, 79, 81, 58,..."
location_name,object,0,38722,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_id,int64,0,38758,2,"[1, 2]"
sex_name,object,0,38758,2,"[Hombre, Mujer]"
age_id,int64,0,38745,15,"[1, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17..."
age_name,object,0,38745,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."


In [10]:
# --- Identify categorical columns from the EDA summary ---

categorical_summary = eda_summary[eda_summary["dtype"] == "object"]

categorical_summary

,dtype,null_count,duplicate_count,n_unique,unique_values
population_group_name,object,0,38759,1,[Toda la población]
measure_name,object,0,38759,1,[Prevalencia]
location_name,object,0,38722,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_name,object,0,38758,2,"[Hombre, Mujer]"
age_name,object,0,38745,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."
cause_name,object,0,38759,1,[Trastornos del espectro autista]
metric_name,object,0,38759,1,[Tasa]


In [11]:
# --- Identify categorical variables with more than one category ---

cat_key_columns = eda_summary[
    (eda_summary["dtype"] == "object") &
    (eda_summary["n_unique"] > 1)
]

cat_key_columns

,dtype,null_count,duplicate_count,n_unique,unique_values
location_name,object,0,38722,38,"[Eslovenia, Turquía, Chile, Colombia, Japón, H..."
sex_name,object,0,38758,2,"[Hombre, Mujer]"
age_name,object,0,38745,15,"[Menores de 5 años, 5-9 años, 10-14 años, 15-1..."


In [12]:
# --- Inspect unique values of key categorical columns ---

for col in cat_key_columns.index:
    print(f"\nUnique values in {col}:")

    values = df[col].unique()

    if col in ["location_name", "sex_name"]:
       values = sorted(values)

    print(values)


Unique values in location_name:
['Alemania', 'Australia', 'Austria', 'Bélgica', 'Canadá', 'Chile', 'Colombia', 'Corea del Sur', 'Costa Rica', 'Dinamarca', 'Eslovaquia', 'Eslovenia', 'España', 'Estados Unidos de América', 'Estonia', 'Finlandia', 'Francia', 'Grecia', 'Holanda', 'Hungría', 'Irlanda', 'Islandia', 'Israel', 'Italia', 'Japón', 'Letonia', 'Lituania', 'Luxemburgo', 'México', 'Noruega', 'Nueva Zelanda', 'Polonia', 'Portugal', 'Reino Unido', 'República Checa', 'Suecia', 'Suiza', 'Turquía']

Unique values in sex_name:
['Hombre', 'Mujer']

Unique values in age_name:
['Menores de 5 años' '5-9 años' '10-14 años' '15-19 años' '20-24 años'
 '25-29 años' '30-34 años' '35-39 años' '40-44 años' '45-49 años'
 '50-54 años' '55-59 años' '60-64 años' '65-69 años' '70+ años']


In [13]:
# --- Define category normalization mappings ---

sex_mapping = {
    "Hombre": "Male",
    "Mujer": "Female"
}

age_mapping = {
    "Menores de 5 años": "< 5",
    "5-9 años": "5-9",
    "10-14 años": "10-14",
    "15-19 años": "15-19",
    "20-24 años": "20-24",
    "25-29 años": "25-29",
    "30-34 años": "30-34",
    "35-39 años": "35-39",
    "40-44 años": "40-44",
    "45-49 años": "45-49",
    "50-54 años": "50-54",
    "55-59 años": "55-59",
    "60-64 años": "60-64",
    "65-69 años": "65-69 ",
    "70+ años": "70 >"
}

location_mapping = {
    "Alemania": "Germany",
    "Australia": "Australia",
    "Austria": "Austria",
    "Bélgica": "Belgium",
    "Canadá": "Canada",
    "Chile": "Chile",
    "Colombia": "Colombia",
    "Corea del Sur": "South Korea",
    "Costa Rica": "Costa Rica",
    "Dinamarca": "Denmark",
    "Eslovaquia": "Slovakia",
    "Eslovenia": "Slovenia",
    "España": "Spain",
    "Estados Unidos de América": "United States",
    "Estonia": "Estonia",
    "Finlandia": "Finland",
    "Francia": "France",
    "Grecia": "Greece",
    "Holanda": "Netherlands",
    "Hungría": "Hungary",
    "Irlanda": "Ireland",
    "Islandia": "Iceland",
    "Israel": "Israel",
    "Italia": "Italy",
    "Japón": "Japan",
    "Letonia": "Latvia",
    "Lituania": "Lithuania",
    "Luxemburgo": "Luxembourg",
    "México": "Mexico",
    "Noruega": "Norway",
    "Nueva Zelanda": "New Zealand",
    "Polonia": "Poland",
    "Portugal": "Portugal",
    "Reino Unido": "United Kingdom",
    "República Checa": "Czech Republic",
    "Suecia": "Sweden",
    "Suiza": "Switzerland",
    "Turquía": "Turkey"
}

In [14]:
# --- Apply category normalization mappings and normalice name of the columns ---

df["gender"] = df["sex_name"].map(sex_mapping)
df["age_range"] = df["age_name"].map(age_mapping)
df["country"] = df["location_name"].map(location_mapping)

In [15]:
# --- Verify normalized categories ---

for col in ["country", "gender", "age_range"]:
    print(f"\nUnique values in {col}:")
    print(df[col].unique())


Unique values in country:
['Slovenia' 'Turkey' 'Chile' 'Colombia' 'Japan' 'Netherlands' 'Austria'
 'Norway' 'Finland' 'Germany' 'Estonia' 'Greece' 'Mexico' 'Hungary'
 'Australia' 'Spain' 'South Korea' 'Costa Rica' 'Slovakia' 'New Zealand'
 'Czech Republic' 'Canada' 'Italy' 'Switzerland' 'Latvia' 'Portugal'
 'United States' 'Belgium' 'Ireland' 'Sweden' 'Iceland' 'Poland'
 'United Kingdom' 'France' 'Israel' 'Lithuania' 'Denmark' 'Luxembourg']

Unique values in gender:
['Male' 'Female']

Unique values in age_range:
['< 5' '5-9' '10-14' '15-19' '20-24' '25-29' '30-34' '35-39' '40-44'
 '45-49' '50-54' '55-59' '60-64' '65-69 ' '70 >']


In [16]:
# --- Rename analytical measure columns for clarity ---

df = df.rename(columns={
    "val": "prevalence",
    "upper": "upper_ui",
    "lower": "lower_ui"
})

In [17]:
# --- Verify renamed analytical columns ---

print("Columns after renaming:")
print(df[["prevalence", "upper_ui", "lower_ui"]].columns.tolist())

print("\nExample values:")
print(df[["prevalence", "upper_ui", "lower_ui"]].head())

Columns after renaming:
['prevalence', 'upper_ui', 'lower_ui']

Example values:
   prevalence     upper_ui    lower_ui
0  939.037586  1855.692963  459.595189
1  363.753608   718.021074  177.685811
2  920.239063  1984.293942  435.700531
3  356.527957   772.392177  168.000119
4  912.201931  1979.073395  358.173398


In [18]:
# --- Build analytical dataset for export ---

analytical_columns = [
    "country",
    "gender",
    "age_range",
    "year",
    "prevalence",
    "upper_ui",
    "lower_ui"
]

df_processed = df[analytical_columns].copy()

In [19]:
# --- Verify processed dataset structure ---

print("Dataset shape:", df_processed.shape)

print("\nColumns:")
print(list(df_processed.columns))

df_processed.head()

Dataset shape: (38760, 7)

Columns:
['country', 'gender', 'age_range', 'year', 'prevalence', 'upper_ui', 'lower_ui']


,country,gender,age_range,year,prevalence,upper_ui,lower_ui
0,Slovenia,Male,< 5,1991,939.037586,1855.692963,459.595189
1,Slovenia,Female,< 5,1991,363.753608,718.021074,177.685811
2,Slovenia,Male,5-9,1991,920.239063,1984.293942,435.700531
3,Slovenia,Female,5-9,1991,356.527957,772.392177,168.000119
4,Slovenia,Male,10-14,1991,912.201931,1979.073395,358.173398


In [20]:
# --- Export processed analytical dataset ---

output_file = PROCESSED_DIR / "gbd_asd_prevalence_oecd_processed.csv"

df_processed.to_csv(output_file, index=False)

print("Dataset exported to:", output_file)

Dataset exported to: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\gbd_asd_prevalence_oecd_processed.csv


In [23]:
# --- Verify exported processed dataset ---

# Load exported dataset to confirm successful write and correct structure
df_check = pd.read_csv(output_file)

print("File exists:", output_file.exists())
print("Shape:", df_check.shape)
print("Columns:", list(df_check.columns))

File exists: True
Shape: (38760, 7)
Columns: ['country', 'gender', 'age_range', 'year', 'prevalence', 'upper_ui', 'lower_ui']
